In [1]:
import torch

from usta_model import UstaModel
from usta_tokenizer import UstaTokenizer

device = "cpu"

if torch.cuda.is_available():
  device = "cuda"
elif torch.backends.mps.is_available():
  device = "mps"
  

print(f"Using device: {device}")

u_tokenizer = UstaTokenizer("tokenizer.json")

prompts = [
  "the capital of the united",
  "madrid is in",
  "the capital of france is",
  "the capital of germany is"
]

tokens = u_tokenizer.encode(prompts[0])
tokens = tokens.to(device)
print(tokens)
batch_tokens = u_tokenizer.encode_batch(prompts, 32)
batch_tokens = batch_tokens.to(device)
batch_tokens.shape

Using device: mps
tensor([ 0, 61,  1, 61,  2, 61,  0, 61,  3], device='mps:0')


torch.Size([4, 32])

In [2]:
torch.manual_seed(1)
context_length = 32

u_model = UstaModel(
  vocab_size=len(u_tokenizer.vocab),
  embedding_dim=12,
  num_heads=4,
  context_length=context_length,
  num_layers=8,
  device=device
)

# load model
u_model.load_state_dict(torch.load("u_model.pth"))

<All keys matched successfully>

In [3]:
out = u_model(batch_tokens)
out.shape

torch.Size([4, 32, 64])

In [4]:
out.flatten(0, 1).shape

torch.Size([128, 64])

In [5]:
# temperature
# top_k 
# top_p

In [6]:
top_k = 10

In [7]:
sorted_outs = sorted(out[-1][-1].tolist(), reverse=True)
sorted_indexes = []
for so in sorted_outs[:top_k]:
  so_index = out[-1][-1].tolist().index(so)
  sorted_indexes.append(so_index)
sorted_outs = torch.tensor(sorted_outs[:top_k])
sorted_outs, sorted_indexes

(tensor([26.3418, 22.2049, 18.7295, 18.4593, 16.5312, 12.3311, 11.8475, 10.8270,
          9.1757,  8.8545]),
 [58, 59, 60, 61, 24, 45, 0, 19, 47, 40])

In [8]:
values, indexes = torch.topk(out[-1][-1], k=10)
values, indexes

(tensor([26.3418, 22.2049, 18.7295, 18.4593, 16.5312, 12.3311, 11.8475, 10.8270,
          9.1757,  8.8545], device='mps:0', grad_fn=<TopkBackward0>),
 tensor([58, 59, 60, 61, 24, 45,  0, 19, 47, 40], device='mps:0'))

In [9]:
temperature = 10.51
adjusted_outs = torch.tensor(sorted_outs) / temperature
adjusted_outs

/var/folders/52/z_f1c0gj4ks99n5rm2trv1jc0000gn/T/ipykernel_3474/2885985782.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  adjusted_outs = torch.tensor(sorted_outs) / temperature


tensor([2.5064, 2.1127, 1.7821, 1.7564, 1.5729, 1.1733, 1.1273, 1.0302, 0.8730,
        0.8425])